In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, expr, regexp_replace, split, size, slice,
    lower, trim, when, year, month, dayofweek, hour, length,
    udf, lit, concat_ws, substring_index, max as spark_max
)
from pyspark.sql.types import *

# ----------------------------------------------------------------------------------
# 1. Spark session + logging
# ----------------------------------------------------------------------------------
spark = (
    SparkSession.builder
        .appName("CleanGitHubCommits")
        .getOrCreate()
)
spark.sparkContext.setLogLevel("INFO")
log = spark._jvm.org.apache.log4j.LogManager.getLogger("CleanGitHubCommits")

In [6]:
# ----------------------------------------------------------------------------------
# 2. Schema – only keep what we actually need
# ----------------------------------------------------------------------------------
file_schema = StructType([
    StructField("filename", StringType(), True),
    StructField("status", StringType(), True),
    StructField("additions", IntegerType(), True),
    StructField("deletions", IntegerType(), True)
])

commit_schema = StructType([
    StructField("sha", StringType(), True),
    StructField("commit", StructType([
        StructField("author", StructType([
            StructField("name", StringType(), True),
            StructField("email", StringType(), True),
            StructField("date", StringType(), True)
        ]), True),
        StructField("committer", StructType([
            StructField("name", StringType(), True),
            StructField("email", StringType(), True),
            StructField("date", StringType(), True)
        ]), True),
        StructField("message", StringType(), True)
    ]), True),
    StructField("html_url", StringType(), True),
    StructField("files", ArrayType(file_schema), True)
])

# ----------------------------------------------------------------------------------
# 3. Load raw Kafka messages
# ----------------------------------------------------------------------------------
df_raw = (
    spark.read
         .format("kafka")
         .option("kafka.bootstrap.servers", "kafka:29092")
         .option("subscribe", "github-commits")
         .option("startingOffsets", "earliest")
         .option("endingOffsets", "latest")
         .load()
)
count = df_raw.count()
print(f"Successfully loaded {count} records from topic 'github-commits'.")



25/07/11 11:49:15 INFO AdminClientConfig: AdminClientConfig values: 
	auto.include.jmx.reporter = true
	bootstrap.servers = [kafka:29092]
	client.dns.lookup = use_all_dns_ips
	client.id = 
	connections.max.idle.ms = 300000
	default.api.timeout.ms = 60000
	metadata.max.age.ms = 300000
	metric.reporters = []
	metrics.num.samples = 2
	metrics.recording.level = INFO
	metrics.sample.window.ms = 30000
	receive.buffer.bytes = 65536
	reconnect.backoff.max.ms = 1000
	reconnect.backoff.ms = 50
	request.timeout.ms = 30000
	retries = 2147483647
	retry.backoff.ms = 100
	sasl.client.callback.handler.class = null
	sasl.jaas.config = null
	sasl.kerberos.kinit.cmd = /usr/bin/kinit
	sasl.kerberos.min.time.before.relogin = 60000
	sasl.kerberos.service.name = null
	sasl.kerberos.ticket.renew.jitter = 0.05
	sasl.kerberos.ticket.renew.window.factor = 0.8
	sasl.login.callback.handler.class = null
	sasl.login.class = null
	sasl.login.connect.timeout.ms = null
	sasl.login.read.timeout.ms = null
	sasl.login.ref

Successfully loaded 31 records from topic 'github-commits'.


25/07/11 11:49:21 INFO MemoryStore: Block broadcast_1_piece0 stored as bytes in memory (estimated size 5.9 KiB, free 434.3 MiB)
25/07/11 11:49:21 INFO BlockManagerInfo: Added broadcast_1_piece0 in memory on aab49221380c:35943 (size: 5.9 KiB, free: 434.4 MiB)
25/07/11 11:49:21 INFO SparkContext: Created broadcast 1 from broadcast at DAGScheduler.scala:1611
25/07/11 11:49:21 INFO DAGScheduler: Submitting 1 missing tasks from ResultStage 2 (MapPartitionsRDD[11] at count at <unknown>:0) (first 15 tasks are for partitions Vector(0))
25/07/11 11:49:21 INFO TaskSchedulerImpl: Adding task set 2.0 with 1 tasks resource profile 0
25/07/11 11:49:21 INFO TaskSetManager: Starting task 0.0 in stage 2.0 (TID 1) (aab49221380c, executor driver, partition 0, NODE_LOCAL, 8999 bytes) 
25/07/11 11:49:21 INFO Executor: Running task 0.0 in stage 2.0 (TID 1)
25/07/11 11:49:21 INFO ShuffleBlockFetcherIterator: Getting 1 (60.0 B) non-empty blocks including 1 (60.0 B) local and 0 (0.0 B) host-local and 0 (0.0 B)

In [ ]:

# Kafka’s `value` is binary; cast to string and parse JSON
df_json = (
    df_raw.select(from_json(col("value").cast("string"), commit_schema).alias("c"))
           .select("c.*")
)

# ----------------------------------------------------------------------------------
# 4. Helper UDFs / expressions
# ----------------------------------------------------------------------------------

# Simple language inference from filename extension
ext2lang = {
    "py": "Python", "java": "Java", "js": "JavaScript", "ts": "TypeScript",
    "cpp": "C++", "c": "C", "cs": "C#", "rb": "Ruby", "go": "Go",
    "rs": "Rust", "kt": "Kotlin", "swift": "Swift", "php": "PHP",
    "scala": "Scala", "md": "Markdown", "yml": "YAML", "yaml": "YAML",
}
@udf("string")
def infer_lang(filename):
    ext = filename.rsplit('.', 1)[-1].lower() if '.' in filename else ""
    return ext2lang.get(ext, "Other")

# Change‑type heuristic
change_expr = (
    when(lower(col("commit_message_clean")).rlike(r"\bfix(e[ds])?\b|bug"), "fix")
    .when(lower(col("commit_message_clean")).rlike(r"\brefactor|clean"), "refactor")
    .when(lower(col("commit_message_clean")).rlike(r"\b(add|feature|implement)"), "feature")
    .otherwise("other")
)

# Discard bot/noreply e‑mails
def strip_bot(email):
    if email is None:
        return None
    e = email.lower()
    if "noreply" in e or "bot@" in e:
        return None
    return e
strip_bot_udf = udf(strip_bot, StringType())

# ----------------------------------------------------------------------------------
# 5. Flatten to file‑level rows
# ----------------------------------------------------------------------------------
df_exp = (
    df_json
      .withColumn("repo",             substring_index(col("html_url"), "/", -2))  # owner/repo
      .withColumn("author_name_raw",  col("commit.author.name"))
      .withColumn("author_email_raw", col("commit.author.email"))
      .withColumn("committer_name",   col("commit.committer.name"))
      .withColumn("committer_email",  col("commit.committer.email"))
      .withColumn("commit_timestamp", col("commit.author.date").cast("timestamp"))
      .withColumn("commit_message_raw", col("commit.message"))
      .withColumn("files",            col("files"))
      .withColumn("author_name",      lower(trim(col("author_name_raw"))))
      .withColumn("author_email",     strip_bot_udf(col("author_email_raw")))
      .drop("author_name_raw", "author_email_raw")
      .withColumn("commit_message_clean",
                  lower(regexp_replace(col("commit_message_raw"), r"http\S+|\p{So}", "")))
      .withColumn("year",        year("commit_timestamp"))
      .withColumn("month",       month("commit_timestamp"))
      .withColumn("day_of_week", dayofweek("commit_timestamp") - 2)  # Spark: 1=Sun
      .withColumn("hour_of_day", hour("commit_timestamp"))
      .withColumn("file",        expr("explode(files)"))
      .selectExpr(
          "repo",
          "sha as commit_sha",
          "author_name",
          "author_email",
          "committer_name",
          "committer_email",
          "file.filename as file_path",
          "file.status as change_status",
          "file.additions as lines_added",
          "file.deletions as lines_deleted",
          "commit_message_clean",
          "commit_timestamp",
          "year","month","day_of_week","hour_of_day"
      )
)

# ----------------------------------------------------------------------------------
# 6. File‑level enrichments
# ----------------------------------------------------------------------------------
df_clean = (
    df_exp
      .withColumn("file_name", split(col("file_path"), "/").getItem(-1))
      .withColumn("path_depth", size(split(col("file_path"), "/")) - 1)
      .withColumn("module_candidate",
                  concat_ws("/",
                            slice(split(col("file_path"), "/"), 1, 2)))  # first 1‑2 folders
      .withColumn("language", infer_lang(col("file_name")))
      .withColumn("commit_size", col("lines_added") + col("lines_deleted"))
      .withColumn("change_type", change_expr)
      .dropDuplicates(["commit_sha", "file_path"])  # defensive
)

# ----------------------------------------------------------------------------------
# 7. Persist or hand off
# ----------------------------------------------------------------------------------
(df_clean
  .write
  .mode("overwrite")
  .parquet("./")
)

log.info(f"Written {df_clean.count()} clean file‑level rows")


AnalysisException: Failed to find data source: kafka. Please deploy the application as per the deployment section of Structured Streaming + Kafka Integration Guide.